In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_38929/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,DraftKings Pick6,player_points,Donovan Mitchell,Over,25.5,-137,2025-11-22,2025-11-22T00:30:44Z
1,DraftKings Pick6,player_points,Donovan Mitchell,Under,25.5,-137,2025-11-22,2025-11-22T00:30:44Z
2,DraftKings Pick6,player_points,Evan Mobley,Over,24.5,-137,2025-11-22,2025-11-22T00:30:44Z
3,DraftKings Pick6,player_points,Evan Mobley,Under,24.5,-137,2025-11-22,2025-11-22T00:30:44Z
4,DraftKings Pick6,player_points,Pascal Siakam,Over,23.5,-137,2025-11-22,2025-11-22T00:30:44Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 139 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
189,Jerami Grant,DraftKings,22.5,17.10,Under,-111,1,6.09,0.676,Med
1060,Bennedict Mathurin,BetMGM,21.5,26.02,Over,110,1,5.92,0.538,High
1140,Tre Jones,BetMGM,9.5,14.89,Over,-110,1,5.71,0.628,Med
854,Alperen Sengun,BetRivers,24.5,28.14,Over,112,0,5.09,0.455,High
444,Keyonte George,FanDuel,18.5,23.11,Over,100,1,5.01,0.501,High


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 110 players...
Processing 101 players...
Generated 4830 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
4199,Jerami Grant,Miles McBride,22.5,8.5,17.10,14.76,0.847,0.834,under,over,1,10.77,0.538,Med,High
2050,Tre Jones,Karl-Anthony Towns,9.5,21.5,14.89,26.94,0.823,0.790,over,over,1,9.12,0.456,Med,High
2641,Klay Thompson,Alperen Sengun,12.5,23.5,8.45,28.14,0.755,0.762,under,over,1,6.92,0.346,Med,High
4242,Buddy Hield,Keyonte George,7.5,18.5,11.51,23.11,0.748,0.750,over,over,1,6.49,0.325,High,High
3369,Dillon Brooks,Lauri Markkanen,18.5,24.5,22.80,28.82,0.743,0.734,over,over,1,6.03,0.301,High,High
1536,Tristan Vukcevic,Isaac Okoro,12.5,7.5,9.12,11.22,0.732,0.733,under,over,0,5.77,0.289,Med,Med
4748,Bogdan Bogdanović,Jordan Clarkson,11.5,9.5,8.22,13.22,0.712,0.728,under,over,0,5.25,0.262,Med,High
3085,Saddiq Bey,Will Richard,8.5,7.5,11.87,10.50,0.700,0.701,over,over,0,4.44,0.222,High,Med
2349,Kevin Huerter,Mikal Bridges,13.5,15.5,16.73,18.74,0.698,0.696,over,over,0,4.29,0.214,High,High
4167,Draymond Green,Tristan da Silva,8.5,12.5,11.35,15.69,0.679,0.695,over,over,0,3.88,0.194,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 134 players...
Processing 122 players...
Generated 7056 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
3889,Tre Jones,Jerami Grant,9.5,22.5,14.89,17.10,over,under,0.823,0.847,0.6827,0.245,0.269,0.362,10.48,0.524,1,5.82,5.28,Med,Med,"(3.5, 26.3)","(6.8, 27.5)",0.05,0,104.8
5841,Alperen Sengun,Karl-Anthony Towns,23.5,21.5,28.14,26.94,over,over,0.762,0.790,0.5900,0.184,0.212,0.268,7.70,0.385,1,6.51,6.74,High,High,"(15.4, 40.9)","(13.7, 40.2)",0.05,0,77.0
4531,Klay Thompson,Julius Randle,12.5,21.5,8.45,26.17,under,over,0.755,0.756,0.5595,0.177,0.178,0.237,6.79,0.339,1,5.86,6.74,Med,High,"(0.0, 19.9)","(13.0, 39.4)",0.05,0,67.9
6510,Buddy Hield,Keyonte George,7.5,18.5,11.51,23.11,over,over,0.748,0.750,0.5497,0.169,0.172,0.227,6.49,0.325,1,6.01,6.82,High,High,"(0.0, 23.3)","(9.7, 36.5)",0.05,0,64.9
5152,Dillon Brooks,Lauri Markkanen,18.5,24.5,22.80,28.82,over,over,0.743,0.734,0.5342,0.165,0.156,0.211,6.03,0.301,1,6.60,6.92,High,High,"(9.9, 35.7)","(15.3, 42.4)",0.05,0,60.3
3962,Isaac Okoro,Aaron Gordon,7.5,16.5,11.22,19.75,over,over,0.733,0.732,0.5262,0.155,0.154,0.203,5.79,0.289,0,5.97,5.24,Med,Med,"(0.0, 22.9)","(9.5, 30.0)",0.05,0,57.9
6035,Cameron Johnson,Jordan Clarkson,12.5,9.5,9.26,13.22,under,over,0.724,0.728,0.5172,0.146,0.150,0.194,5.52,0.276,0,5.44,6.12,Med,High,"(0.0, 19.9)","(1.2, 25.2)",0.05,0,55.2
6561,Will Richard,Bogdan Bogdanović,7.5,11.5,10.50,8.22,over,under,0.701,0.712,0.4895,0.123,0.134,0.165,4.68,0.234,0,5.67,5.86,Med,Med,"(0.0, 21.6)","(0.0, 19.7)",0.05,0,46.8
3365,Kevin Huerter,Saddiq Bey,13.5,8.5,16.73,11.87,over,over,0.698,0.700,0.4793,0.120,0.122,0.155,4.38,0.219,0,6.21,6.41,High,High,"(4.6, 28.9)","(0.0, 24.4)",0.05,0,43.8
6969,Brandon Miller,Mikal Bridges,14.5,15.5,12.34,18.74,under,over,0.686,0.696,0.4680,0.108,0.118,0.143,4.04,0.202,0,4.45,6.33,Low,High,"(3.6, 21.1)","(6.3, 31.2)",0.05,0,40.4


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 110 players...
Processing 101 players...
Generated 165292 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
92425,Tre Jones,Jerami Grant,Miles McBride,9.5,22.5,8.5,14.89,17.10,14.76,0.823,0.847,0.834,over,under,over,1,21.39,0.428,Med,Med,High
114933,Klay Thompson,Alperen Sengun,Karl-Anthony Towns,12.5,23.5,21.5,8.45,28.14,26.94,0.755,0.762,0.790,under,over,over,1,14.56,0.291,Med,High,High
137548,Dillon Brooks,Buddy Hield,Keyonte George,18.5,7.5,18.5,22.80,11.51,23.11,0.743,0.748,0.750,over,over,over,1,12.49,0.250,High,High,High
71721,Tristan Vukcevic,Isaac Okoro,Lauri Markkanen,12.5,7.5,24.5,9.12,11.22,28.82,0.732,0.733,0.734,under,over,over,0,11.27,0.225,Med,Med,High
161108,Will Richard,Bogdan Bogdanović,Jordan Clarkson,7.5,11.5,9.5,10.50,8.22,13.22,0.701,0.712,0.728,over,under,over,0,9.65,0.193,Med,Med,High


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 134 players...
Processing 122 players...
Generated 292757 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
204679,Tre Jones,Jerami Grant,Karl-Anthony Towns,9.5,22.5,21.5,14.89,17.10,26.94,0.823,0.847,0.790,over,under,over,1,19.73,0.395,Med,Med,High
229569,Klay Thompson,Julius Randle,Alperen Sengun,12.5,21.5,23.5,8.45,26.17,28.14,0.755,0.756,0.762,under,over,over,1,13.49,0.270,Med,High,High
251395,Dillon Brooks,Buddy Hield,Keyonte George,18.5,7.5,18.5,22.80,11.51,23.11,0.743,0.748,0.750,over,over,over,1,12.49,0.250,High,High,High
207579,Isaac Okoro,Aaron Gordon,Lauri Markkanen,7.5,16.5,24.5,11.22,19.75,28.82,0.733,0.732,0.734,over,over,over,0,11.28,0.226,Med,Med,High
276094,Cameron Johnson,Bogdan Bogdanović,Jordan Clarkson,12.5,11.5,9.5,9.26,8.22,13.22,0.724,0.712,0.728,under,under,over,0,10.29,0.206,Med,Med,High


In [10]:
# playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)